In [1]:
from openai import OpenAI

client = OpenAI(
    base_url="https://integrate.api.nvidia.com/v1",
    api_key="nvapi-xxxxxxxxxxxxxxxxxxxx"
)

In [2]:
def filter_links_with_llm(current_url, links):
    prompt = f"""You are a smart web scraping agent. Go to various links of different departments too to make a valid corpus. The final corpus formed should be all in paragraph format only. After seeing the tables or images, you only have to translate it into paragraph format. Actually my question stated - Students are required to collect textual data from at least three of the following suggestive
sources: IIT Jodhpur official website pages (departments, academic programs, research pages, announcements), Academic regulation documents (must), Institute newsletters or circulars,
Faculty profile pages, Course syllabus, etc. (Only consider the English Text, remove text from other languages, if any.).

From the list of URLs below, select ONLY those relevant for building an academic corpus:
- departments
- faculty profiles
- research
- academics
- syllabus
- announcements
- regulations

Ignore:
- login
- contact
- social media
- irrelevant navigation

Return ONLY valid URLs (one per line).

Current page: {current_url}

Links:
{links}
"""

    response = client.chat.completions.create(
        model="mistralai/mistral-nemotron",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.6,
        max_tokens=4096,
    )

    text = response.choices[0].message.content
    return [line.strip() for line in text.split("\n") if line.startswith("http")]

In [3]:
def extract_clean_text_with_llm(html_text):
    prompt = f"""
Extract only meaningful academic text from this webpage.

Keep:
- course details
- research content
- faculty info
- announcements
- academic descriptions

Remove:
- menus
- headers/footers
- repeated navigation text
- garbage text

Return clean paragraph text only.

HTML:
{html_text[:15000]}
"""

    response = client.chat.completions.create(
        model="mistralai/mistral-nemotron",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.6,
        max_tokens=4096,
    )

    return response.choices[0].message.content

In [4]:
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin, urlparse
from collections import deque

BASE_DOMAIN = "iitj.ac.in"

visited = set()
queue = deque([
    "https://www.iitj.ac.in/",
    "https://www.iitj.ac.in/academics",
    "https://www.iitj.ac.in/research"
])

corpus = []

def is_valid(url):
    return BASE_DOMAIN in urlparse(url).netloc

while queue and len(visited) < 50:   
    url = queue.popleft()
    if url in visited:
        continue
    print("Visiting:", url)

    try:
        response = requests.get(url, timeout=10)
        soup = BeautifulSoup(response.text, "lxml")
        links = [urljoin(url, a['href']) for a in soup.find_all("a", href=True)]
        filtered_links = filter_links_with_llm(url, links)

        for link in filtered_links:
            if is_valid(link) and link not in visited:
                queue.append(link)
        clean_text = extract_clean_text_with_llm(response.text)

        if clean_text:
            corpus.append(clean_text)
        visited.add(url)

    except Exception as e:
        print("Error:", e)

Visiting: https://www.iitj.ac.in/
Visiting: https://www.iitj.ac.in/academics
Visiting: https://www.iitj.ac.in/research
Visiting: https://www.iitj.ac.in/m/Index/main-departments?lg=en
Visiting: https://www.iitj.ac.in/office-of-academics/en/academics
Visiting: https://www.iitj.ac.in/m/Index/main-programs?lg=en
Visiting: https://www.iitj.ac.in/admission-postgraduate-programs/en/Admission-to-Postgraduate-Programs
Visiting: https://www.iitj.ac.in/main/en/admission-links
Visiting: https://www.iitj.ac.in/office-of-research-development/en/office-of-research-and-development
Visiting: https://www.iitj.ac.in/main/en/faculty-members
Visiting: https://www.iitj.ac.in/main/en/scholars-in-residence
Visiting: https://www.iitj.ac.in/main/en/adjunct-faculty-members
Visiting: https://www.iitj.ac.in/main/en/visiting-faculty-members
Visiting: https://www.iitj.ac.in/main/en/research-highlight
Visiting: https://www.iitj.ac.in/main/en/all-announcement
Visiting: https://www.iitj.ac.in/main/en/recruitments
Visit

In [5]:
with open("agentic_corpus.txt", "w", encoding="utf-8") as f:
    for doc in corpus:
        f.write(doc + "\n\n")